# Explore Pinterest & CiteULike NPZ Datasets
Notebook này giúp bạn:
- Đọc dữ liệu từ file `.npz`
- In sample row của train và test
- Tính thống kê cơ bản
- Vẽ biểu đồ phân phối tương tác user/item và top item phổ biến

In [ ]:
from pathlib import Path
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')

DATASETS = {
    'Pinterest': Path('dataset/pinterest.npz/pinterest.npz'),
    'CiteULike-A': Path('dataset/citeulike-a.npz/citeulike-a.npz'),
}

TOP_K = 15
SAMPLE_TRAIN_ROWS = 8
SAMPLE_TEST_USERS = 5

In [ ]:
def load_npz_dataset(file_path: Path):
    data = np.load(file_path, allow_pickle=True)
    train = data['train_data']

    test_dict = {}
    if 'test_data' in data.files:
        raw_test = data['test_data']
        if isinstance(raw_test, np.ndarray) and raw_test.shape == ():
            test_dict = raw_test.item()
        elif isinstance(raw_test, dict):
            test_dict = raw_test

    return train, test_dict


def summarize_train(train_data: np.ndarray):
    users = train_data[:, 0].astype(int)
    items = train_data[:, 1].astype(int)

    user_counts = Counter(users.tolist())
    item_counts = Counter(items.tolist())
    pair_counts = Counter(zip(users.tolist(), items.tolist()))

    return {
        'n_interactions': int(train_data.shape[0]),
        'n_users': len(user_counts),
        'n_items': len(item_counts),
        'user_counts': user_counts,
        'item_counts': item_counts,
        'duplicate_pairs': sum(v - 1 for v in pair_counts.values() if v > 1),
    }


def print_train_test_samples(name, train_data, test_dict, n_train=8, n_test=5):
    print('=' * 90)
    print(f'Dataset: {name}')
    print('-' * 90)

    print(f'Sample train rows (first {n_train}):')
    print('  user_id  item_id')
    for row in train_data[:n_train]:
        print(f'  {int(row[0]):7d}  {int(row[1]):7d}')

    print('')
    print(f'Sample test entries (first {n_test} users):')
    if not test_dict:
        print('  (No test_data found)')
        return

    for idx, (u, payload) in enumerate(test_dict.items()):
        if idx >= n_test:
            break
        if isinstance(payload, (tuple, list)) and len(payload) >= 2:
            pos_item = int(payload[0])
            neg_items = payload[1]
            neg_preview = list(neg_items[:10]) if hasattr(neg_items, '__getitem__') else []
            print(
                f'  user={int(u)} | pos={pos_item} | #neg={len(neg_items)} | neg_preview={neg_preview}'
            )
        else:
            print(f'  user={int(u)} | payload={payload}')


def plot_dataset_distributions(name, stats, top_k=15):
    user_vals = np.array(list(stats['user_counts'].values()))
    item_vals = np.array(list(stats['item_counts'].values()))

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f'{name} - Distribution Overview', fontsize=14, fontweight='bold')

    axes[0, 0].hist(user_vals, bins=40, color='#2E86AB', alpha=0.9)
    axes[0, 0].set_title('Interactions per User')
    axes[0, 0].set_xlabel('Interactions')
    axes[0, 0].set_ylabel('Frequency')

    axes[0, 1].hist(item_vals, bins=40, color='#F18F01', alpha=0.9)
    axes[0, 1].set_title('Interactions per Item')
    axes[0, 1].set_xlabel('Interactions')
    axes[0, 1].set_ylabel('Frequency')

    top_users = stats['user_counts'].most_common(top_k)
    axes[1, 0].bar([str(u) for u, _ in top_users], [c for _, c in top_users], color='#6A994E')
    axes[1, 0].set_title(f'Top {top_k} Active Users')
    axes[1, 0].set_xlabel('User ID')
    axes[1, 0].set_ylabel('Interactions')
    axes[1, 0].tick_params(axis='x', rotation=75)

    top_items = stats['item_counts'].most_common(top_k)
    axes[1, 1].bar([str(i) for i, _ in top_items], [c for _, c in top_items], color='#BC4749')
    axes[1, 1].set_title(f'Top {top_k} Popular Items')
    axes[1, 1].set_xlabel('Item ID')
    axes[1, 1].set_ylabel('Interactions')
    axes[1, 1].tick_params(axis='x', rotation=75)

    plt.tight_layout()
    plt.show()

In [ ]:
all_stats = {}

for name, path in DATASETS.items():
    if not path.exists():
        print(f'[WARN] File not found: {path}')
        continue

    train_data, test_dict = load_npz_dataset(path)
    stats = summarize_train(train_data)
    all_stats[name] = stats

    density = stats['n_interactions'] / (stats['n_users'] * stats['n_items'])

    print('=' * 90)
    print(f'Dataset: {name}')
    print(f'Path: {path}')
    print('-' * 90)
    print(f'Train interactions : {stats["n_interactions"]:,}')
    print(f'Unique users       : {stats["n_users"]:,}')
    print(f'Unique items       : {stats["n_items"]:,}')
    print(f'Density            : {density:.8f}')
    print(f'Duplicate pairs    : {stats["duplicate_pairs"]:,}')
    print(f'Test users         : {len(test_dict):,}')

    print_train_test_samples(
        name=name,
        train_data=train_data,
        test_dict=test_dict,
        n_train=SAMPLE_TRAIN_ROWS,
        n_test=SAMPLE_TEST_USERS,
    )

In [ ]:
for name, path in DATASETS.items():
    if not path.exists():
        continue
    train_data, _ = load_npz_dataset(path)
    stats = summarize_train(train_data)
    plot_dataset_distributions(name, stats, top_k=TOP_K)

In [ ]:
if all_stats:
    names = list(all_stats.keys())
    interactions = [all_stats[n]['n_interactions'] for n in names]
    users = [all_stats[n]['n_users'] for n in names]
    items = [all_stats[n]['n_items'] for n in names]

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle('Dataset Comparison', fontsize=14, fontweight='bold')

    axes[0].bar(names, interactions, color=['#355070', '#6D597A'])
    axes[0].set_title('Train Interactions')
    axes[0].tick_params(axis='x', rotation=20)

    axes[1].bar(names, users, color=['#B56576', '#E56B6F'])
    axes[1].set_title('Unique Users')
    axes[1].tick_params(axis='x', rotation=20)

    axes[2].bar(names, items, color=['#EAAC8B', '#A7C957'])
    axes[2].set_title('Unique Items')
    axes[2].tick_params(axis='x', rotation=20)

    plt.tight_layout()
    plt.show()
else:
    print('No dataset found to compare.')